# Marketing Attribution and ROI

Generates the final marketing effectiveness output
`data_output/activity_performance_with_roi.csv` from
`../_2_feature_engineering+momentum/data_output/valid_bookings_for_marketing.parquet`
and `data_output/master_marketing_activties.csv`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent
BOOKINGS_PATH = BASE_DIR / "_2_feature_engineering+momentum" / "data_output" / "valid_bookings_for_marketing.parquet"
MARKETING_PATH = Path("data_output") / "master_marketing_activties.csv"
OUT_DIR = Path("data_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

master_bookings = pd.read_parquet(BOOKINGS_PATH)
marketing_activities = pd.read_csv(MARKETING_PATH)


## 1. Attribute Bookings to Marketing Windows


In [ ]:
bookings = master_bookings.copy()
if "booking_date" in bookings.columns:
    bookings["booking_datetime"] = pd.to_datetime(bookings["booking_date"], errors="coerce")
else:
    bookings["booking_datetime"] = pd.to_datetime(bookings["date"], errors="coerce")

if "revenue_thb" not in bookings.columns and "revenue" in bookings.columns:
    bookings["revenue_thb"] = pd.to_numeric(bookings["revenue"], errors="coerce")
if "total_guests" not in bookings.columns and "party_size" in bookings.columns:
    bookings["total_guests"] = bookings["party_size"]

bookings["restaurant_id"] = pd.to_numeric(bookings["restaurant_id"], errors="coerce")
bookings = bookings.dropna(subset=["restaurant_id", "booking_datetime"]).copy()

mkt = marketing_activities.copy()
mkt["restaurant_id"] = (
    pd.to_numeric(mkt.get("crm_restaurant_id"), errors="coerce")
    .combine_first(pd.to_numeric(mkt.get("kol_restaurant_id"), errors="coerce"))
    .combine_first(pd.to_numeric(mkt.get("fb_restaurant_id"), errors="coerce"))
)
mkt["activity_start"] = pd.to_datetime(mkt["activity_start"], errors="coerce").dt.normalize()
mkt["activity_end"] = pd.to_datetime(mkt["activity_end"], errors="coerce").dt.normalize()
mkt["activity_end"] = mkt["activity_end"].fillna(
    mkt["activity_start"] + pd.to_timedelta(pd.to_numeric(mkt.get("fb_campaign_duration_days"), errors="coerce"), unit="D")
)
mkt["activity_end"] = mkt["activity_end"].fillna(pd.Timestamp.today().normalize())
mkt = mkt.dropna(subset=["activity_id", "restaurant_id", "activity_start", "activity_end"]).copy()
mkt = mkt.loc[mkt["activity_end"] >= mkt["activity_start"]].copy()

mkt_exposure = mkt[[
    "activity_id", "channel", "restaurant_id",
    "activity_start", "activity_end",
    "crm_campaign_name", "crm_topic", "crm_audience",
    "kol_platform", "kol_username", "kol_post_url",
    "fb_campaign", "fb_amount_spent_thb",
]].copy()

cand = bookings[["id", "restaurant_id", "booking_datetime", "revenue_thb", "total_guests"]].merge(
    mkt_exposure,
    on="restaurant_id",
    how="left",
    suffixes=("_booking", "_mkt"),
)
attrib = cand[
    (cand["booking_datetime"] >= cand["activity_start"])
    & (cand["booking_datetime"] <= cand["activity_end"])
].copy()
attrib["time_from_start_hours"] = (attrib["booking_datetime"] - attrib["activity_start"]).dt.total_seconds() / 3600.0
attrib["abs_time_from_start_hours"] = attrib["time_from_start_hours"].abs()
attrib = attrib.sort_values(["id", "abs_time_from_start_hours"])
attrib_1 = attrib.drop_duplicates(subset=["id"], keep="first").copy()

print("Bookings:", bookings.shape)
print("Exposures:", mkt_exposure.shape)
print("Attributed bookings:", attrib_1.shape)
attrib_1.head()


## 2. Calculate Lift per Activity


In [ ]:
mkt_lift = mkt_exposure.copy()
mkt_lift["window_hours"] = (mkt_lift["activity_end"] - mkt_lift["activity_start"]).dt.total_seconds() / 3600.0
mkt_lift = mkt_lift.loc[mkt_lift["window_hours"] > 0].copy()
mkt_lift["baseline_start"] = mkt_lift["activity_start"] - pd.to_timedelta(mkt_lift["window_hours"], unit="h")
mkt_lift["baseline_end"] = mkt_lift["activity_start"]

cand_during = bookings[["id", "restaurant_id", "booking_datetime"]].merge(
    mkt_lift[["activity_id", "restaurant_id", "activity_start", "activity_end"]],
    on="restaurant_id",
    how="inner",
)
during_counts = cand_during[
    (cand_during["booking_datetime"] >= cand_during["activity_start"])
    & (cand_during["booking_datetime"] <= cand_during["activity_end"])
].groupby("activity_id").size().rename("bookings_during")

cand_baseline = bookings[["id", "restaurant_id", "booking_datetime"]].merge(
    mkt_lift[["activity_id", "restaurant_id", "baseline_start", "baseline_end"]],
    on="restaurant_id",
    how="left",
)
baseline = cand_baseline[
    (cand_baseline["booking_datetime"] >= cand_baseline["baseline_start"])
    & (cand_baseline["booking_datetime"] < cand_baseline["baseline_end"])
].copy()
baseline_counts = baseline.groupby("activity_id").size().rename("bookings_baseline")

lift_table = pd.concat([during_counts, baseline_counts], axis=1).fillna(0).reset_index()
lift_table["lift"] = lift_table["bookings_during"] - lift_table["bookings_baseline"]

mkt_meta = mkt_lift[[
    "activity_id", "channel", "restaurant_id", "activity_start", "activity_end", "window_hours",
    "crm_campaign_name", "crm_topic", "crm_audience",
    "kol_platform", "kol_username", "kol_post_url",
    "fb_campaign", "fb_amount_spent_thb",
]].drop_duplicates("activity_id")

activity_perf = lift_table.merge(mkt_meta, on="activity_id", how="left")
activity_perf["window_days"] = activity_perf["window_hours"] / 24.0
activity_perf["lift_per_day"] = activity_perf["lift"] / activity_perf["window_days"]
activity_perf["fb_amount_spent_thb"] = pd.to_numeric(activity_perf["fb_amount_spent_thb"], errors="coerce")
activity_perf["cost_per_incremental_booking"] = np.where(
    (activity_perf["channel"] == "FB")
    & activity_perf["lift"].gt(0)
    & activity_perf["fb_amount_spent_thb"].gt(0),
    activity_perf["fb_amount_spent_thb"] / activity_perf["lift"],
    np.nan,
)

activity_perf.head()


## 3. Channel Summary


In [ ]:
channel_summary = (
    activity_perf
    .groupby("channel", as_index=False)
    .agg(
        activities=("activity_id", "nunique"),
        total_bookings_during=("bookings_during", "sum"),
        total_baseline=("bookings_baseline", "sum"),
        total_lift=("lift", "sum"),
        avg_lift=("lift", "mean"),
        median_lift=("lift", "median"),
    )
)
channel_summary["lift_rate_vs_baseline"] = np.where(
    channel_summary["total_baseline"] > 0,
    channel_summary["total_lift"] / channel_summary["total_baseline"],
    np.nan,
)
channel_summary


## 4. Incremental Revenue and ROI


In [ ]:
revenue_data = attrib_1.groupby("activity_id").agg(total_campaign_revenue=("revenue_thb", "sum")).reset_index()
activity_perf = activity_perf.merge(revenue_data, on="activity_id", how="left")
activity_perf["total_campaign_revenue"] = activity_perf["total_campaign_revenue"].fillna(0)
activity_perf["aov_thb"] = np.where(
    activity_perf["bookings_during"] > 0,
    activity_perf["total_campaign_revenue"] / activity_perf["bookings_during"],
    0,
)
activity_perf["incremental_revenue_thb"] = activity_perf["lift"] * activity_perf["aov_thb"]
activity_perf["fb_amount_spent_thb"] = pd.to_numeric(activity_perf["fb_amount_spent_thb"], errors="coerce").fillna(0)
activity_perf["roi"] = np.where(
    (activity_perf["channel"] == "FB") & (activity_perf["fb_amount_spent_thb"] > 0),
    (activity_perf["incremental_revenue_thb"] - activity_perf["fb_amount_spent_thb"]) / activity_perf["fb_amount_spent_thb"],
    np.nan,
)
activity_perf["roi_percentage"] = activity_perf["roi"] * 100
activity_perf.to_csv(OUT_DIR / "activity_performance_with_roi.csv", index=False)
activity_perf.sort_values("roi", ascending=False).head(10)


## 5. Diagnostic Charts


In [ ]:
plot_data = activity_perf.dropna(subset=["roi"]).copy()
if not plot_data.empty:
    top_roi = plot_data.sort_values("roi", ascending=False).head(10)
    top_revenue = plot_data.sort_values("incremental_revenue_thb", ascending=False).head(10)
    plt.style.use("seaborn-v0_8-whitegrid")

    plt.figure(figsize=(10, 6))
    bars1 = plt.barh(top_roi["activity_id"].astype(str), top_roi["roi_percentage"], color="#4C72B0")
    plt.title("Top 10 Campaigns by Marketing ROI (%)")
    plt.xlabel("Return on Investment (%)")
    plt.ylabel("Campaign (Activity ID)")
    plt.gca().invert_yaxis()
    for bar in bars1:
        width = bar.get_width()
        plt.text(width + (width * 0.01), bar.get_y() + bar.get_height() / 2, f"{width:.0f}%", va="center", fontsize=10)
    plt.axvline(0, color="black", linewidth=1)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 6))
    bars2 = plt.barh(top_revenue["activity_id"].astype(str), top_revenue["incremental_revenue_thb"], color="#55A868")
    plt.title("Top 10 Campaigns by Incremental Revenue (THB)")
    plt.xlabel("Incremental Revenue Generated (THB)")
    plt.ylabel("Campaign (Activity ID)")
    plt.gca().invert_yaxis()
    for bar in bars2:
        width = bar.get_width()
        plt.text(width + (width * 0.01), bar.get_y() + bar.get_height() / 2, f"฿{width:,.0f}", va="center", fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("No ROI rows available for plotting.")
